In [1]:
import os
import zipfile
import glob
import pandas as pd
import sys

sys.path.insert(0, r"C:\Users\mjbou\governance-framework\src")
from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources

# --- Version config: update when new release available ---
VDEM_VERSION    = "16"
VDEM_AS_OF_DATE = "2026-03"

VDEM_FILENAME = f"vdem_full_v{VDEM_VERSION}.csv"
VDEM_PATH     = os.path.join(RAW_DIR, VDEM_FILENAME)

# --- Check file exists ---
if os.path.exists(VDEM_PATH):
    size_mb = os.path.getsize(VDEM_PATH) / (1024 * 1024)
    print(f"Found: {VDEM_FILENAME} ({size_mb:.1f} MB)")
else:
    print(f"FILE NOT FOUND: {VDEM_PATH}")
    print("See docs/instructions_data_maintenance.md — VDEM section")

Found: vdem_full_v16.csv (387.5 MB)


## V-Dem Pipeline

**Download instructions:** See `docs/instructions_data_maintenance.md` — VDEM section.

Once the file is in `data/raw/` and named `vdem_full_v{VERSION}.csv`, run the cells below.

In [2]:
matches = glob.glob(os.path.join(DOWNLOADS_DIR, "V-Dem-CY-FullOthers-v16*.zip"))

if not matches:
    print("ZIP not found")
else:
    zip_path = matches[0]
    print(f"Found: {zip_path}")
    with zipfile.ZipFile(zip_path, 'r') as z:
        print("Contents:")
        for f in z.namelist():
            print(f"  {f}")

Found: C:\Users\mjbou\Downloads\V-Dem-CY-FullOthers-v16_csv.zip
Contents:
  V-Dem-CY-Full+Others-v16.csv
  cautionary_notes.pdf
  codebook.pdf
  suggested_citation.pdf
  whats_new.pdf


In [3]:
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extract("V-Dem-CY-Full+Others-v16.csv", RAW_DIR)

# Rename to our standard convention
src = os.path.join(RAW_DIR, "V-Dem-CY-Full+Others-v16.csv")
dst = os.path.join(RAW_DIR, VDEM_FILENAME)
os.rename(src, dst)

# Confirm
size_mb = os.path.getsize(dst) / (1024 * 1024)
print(f"Extracted and renamed to: {VDEM_FILENAME}")
print(f"Size: {size_mb:.1f} MB")

FileExistsError: [WinError 183] Cannot create a file when that file already exists: 'C:\\Users\\mjbou\\governance-framework\\data\\raw\\V-Dem-CY-Full+Others-v16.csv' -> 'C:\\Users\\mjbou\\governance-framework\\data\\raw\\vdem_full_v16.csv'

In [4]:
from datetime import datetime

update_entry(
    "VDEM",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=VDEM_AS_OF_DATE,
    local_filename=VDEM_FILENAME,
    latest_available_version=f"v{VDEM_VERSION}",
    notes="Full+Others CSV. Extracted from ZIP. Manual download via email form."
)

print_entry("VDEM")

[download_log] Updated entry for VDEM
  source_id: VDEM
  last_attempted_date: 2026-05-19
  last_successful_download_date: 2026-05-19
  data_as_of_date: 2026-03
  local_filename: vdem_full_v16.csv
  latest_available_version: v16
  no_update_reason: nan
  notes: Full+Others CSV. Extracted from ZIP. Manual download via email form.


In [5]:
vdem = pd.read_csv(os.path.join(RAW_DIR, VDEM_FILENAME), low_memory=False)
print(f"Shape: {vdem.shape}")
print(f"Columns (first 20): {list(vdem.columns[:20])}")
print(f"Years: {vdem['year'].min()} — {vdem['year'].max()}")
print(f"Countries: {vdem['country_name'].nunique()}")

Shape: (28092, 4618)
Columns (first 20): ['country_name', 'country_text_id', 'country_id', 'year', 'historical_date', 'project', 'historical', 'histname', 'codingstart', 'codingend', 'codingstart_contemp', 'codingend_contemp', 'codingstart_hist', 'codingend_hist', 'gapstart1', 'gapstart2', 'gapstart3', 'gapend1', 'gapend2', 'gapend3']
Years: 1789 — 2025
Countries: 202


In [6]:
# All V-Dem variables used in the framework
VDEM_VARS = [
    # Identifiers
    'country_name', 'country_text_id', 'country_id', 'year',

    # Political settlement
    'v2pepwrses', 'v2pepwrsoc', 'v2x_egal', 'v2psoppaut',

    # Political stability
    'v2x_regime',

    # State capacity
    'v2svstterr', 'v2svdomaut',

    # Government effectiveness
    'v2clrspct',

    # State control over economy
    'v2clstown',

    # Legislative checks
    'v2xlg_legcon', 'v2lgoppart', 'v2lgqstexp', 'v2lginvstp', 'v2lgotovst', 'v2x_horacc',

    # Judicial independence
    'v2juhcind', 'v2juncind', 'v2jucomp', 'v2jupack', 'v2jupurge',

    # Electoral process
    'v2x_polyarchy', 'v2elfrfair', 'v2elirreg', 'v2elintim', 'v2elvotbuy', 'v2elaccept',

    # Political participation
    'v2x_partip', 'v2psprlnks', 'v2pscohesv', 'v2cseeorgs', 'v2dlconslt', 'v2csreprss',

    # Civil liberties
    'v2x_civlib', 'v2x_clpriv', 'v2clrelig', 'v2cldmovem', 'v2cldmovew',
    'v2clsocgrp', 'v2clslavef',

    # Media freedom
    'v2x_freexp_altinf', 'v2mecenefm', 'v2meharjrn', 'v2mecorrpt', 'v2meslfcen',
    'v2merange', 'v2mebias', 'v2mecrit',

    # Civil society space
    'v2cscnsult', 'v2csprtcpt',

    # Government transparency
    'v2cltrnslw',

    # Legal quality and predictability
    'v2clacjstm', 'v2clacjstw', 'v2xeg_eqaccess',

    # Personal security
    'v2cltort', 'v2clkill', 'v2clrgunev',

    # Property rights
    'v2clprptym', 'v2clprptyw', 'v2xcl_prpty',

    # Corruption
    'v2x_corr', 'v2excrptps', 'v2exembez', 'v2lgcrrpt', 'v2jucorrdc',
]

# Check for any variables not found in dataset
missing = [v for v in VDEM_VARS if v not in vdem.columns]
if missing:
    print(f"Missing variables ({len(missing)}):")
    for v in missing:
        print(f"  {v}")
else:
    print("All variables found.")

All variables found.


In [7]:
vdem_filtered = vdem[VDEM_VARS].copy()
vdem_filtered = vdem_filtered[vdem_filtered['year'] >=FRAMEWORK_START_YEAR]

print(f"Shape after filtering: {vdem_filtered.shape}")
print(f"Years: {vdem_filtered['year'].min()} — {vdem_filtered['year'].max()}")
print(f"Countries: {vdem_filtered['country_name'].nunique()}")
print(f"\nMissing values (%):")
missing_pct = (vdem_filtered.isnull().sum() / len(vdem_filtered) * 100).round(1)
print(missing_pct[missing_pct > 0].sort_values(ascending=False))

Shape after filtering: (6383, 68)
Years: 1990 — 2025
Countries: 181

Missing values (%):
v2elfrfair      9.2
v2elaccept      9.2
v2elvotbuy      9.2
v2elintim       9.2
v2elirreg       9.2
v2lgcrrpt       4.0
v2lgoppart      4.0
v2lgqstexp      4.0
v2svstterr      3.3
v2psoppaut      2.6
v2lginvstp      1.5
v2xlg_legcon    1.5
v2lgotovst      1.1
v2jupurge       0.4
v2jupack        0.4
v2jucomp        0.4
v2x_corr        0.4
v2juhcind       0.2
v2x_partip      0.2
v2excrptps      0.2
v2exembez       0.2
v2jucorrdc      0.2
dtype: float64


In [8]:
output_path = os.path.join(PROCESSED_DIR, "vdem_filtered.csv")
vdem_filtered.to_csv(output_path, index=False)
print(f"Written: {output_path}")
print(f"Shape: {vdem_filtered.shape}")

Written: C:\Users\mjbou\governance-framework\data\processed\vdem_filtered.csv
Shape: (6383, 68)


In [9]:
# Remove original extracted filename if it exists
original = os.path.join(RAW_DIR, "V-Dem-CY-Full+Others-v16.csv")
if os.path.exists(original):
    os.remove(original)
    print(f"Removed: {original}")